# 05 — Model Evaluation
**Trio-Based Variant Pathogenicity Classifier**

This notebook:
1. Loads the best saved pipeline from `models/`
2. Builds a hold-out test set from the **last 20 % of trios** (by `trio_id` sort order)
3. Produces all diagnostic plots and saves them to `reports/`
4. Summarises performance per `inheritance_mode`

> Run `notebooks/04_modelling.ipynb` first to produce `models/best_pipeline.joblib`.

## 1 — Imports & Configuration

In [ ]:
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import subprocess
import shap
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

# ── project root on sys.path ──────────────────────────────────────────────────
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import build_features
from src.trio_loader import TrioLoader

# ── constants ─────────────────────────────────────────────────────────────────
DATA_DIR            = ROOT / "data"
MODEL_PATH          = ROOT / "models" / "best_pipeline.joblib"
REPORTS_DIR         = ROOT / "reports"
TRIOS_PATH          = DATA_DIR / "trios.tsv"
PHENOTYPES_PATH     = DATA_DIR / "phenotypes.tsv"
TEST_TRIO_FRACTION  = 0.20   # last 20 % of trios by sort order

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

warnings.filterwarnings("ignore", category=UserWarning)
print(f"shap version : {shap.__version__}")
print(f"ROOT         : {ROOT}")

## 2 — Load Pipeline and Full Variant DataFrame

In [ ]:
# ── Load the serialised pipeline ─────────────────────────────────────────────
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Pipeline not found at {MODEL_PATH}.\n"
        "Run notebooks/04_modelling.ipynb first to train and save a model."
    )

pipeline = joblib.load(MODEL_PATH)
print(f"Loaded pipeline: {pipeline}")

# ── Load all trios via TrioLoader ─────────────────────────────────────────────
loader = TrioLoader(
    trios_path=str(TRIOS_PATH),
    phenotypes_path=str(PHENOTYPES_PATH),
    data_dir=str(DATA_DIR),
)
df_all = loader.load_all()
print(f"\nFull dataset : {df_all.shape[0]:,} variants across {df_all['trio_id'].nunique()} trios")
print(df_all["Classification"].value_counts().to_string())

## 3 — Prepare Hold-Out Test Set

The **last 20 % of trios** (sorted by `trio_id`, i.e. chronological / alphabetical order)  
are held out for evaluation. This mirrors a realistic temporal split where new patients  
arrive sequentially.

In [ ]:
# ── Identify hold-out trios ───────────────────────────────────────────────────
all_trios    = sorted(df_all["trio_id"].unique())
n_test_trios = max(1, int(np.ceil(len(all_trios) * TEST_TRIO_FRACTION)))
test_trios   = all_trios[-n_test_trios:]
train_trios  = all_trios[:-n_test_trios]

print(f"Total trios  : {len(all_trios)}")
print(f"Train trios  : {len(train_trios)} → {train_trios}")
print(f"Test  trios  : {len(test_trios)}  → {test_trios}")

# ── Build test feature matrix ─────────────────────────────────────────────────
df_test = df_all[df_all["trio_id"].isin(test_trios)].copy()
X_test, y_test = build_features(df_test)

# Align the hold-out features to the fitted pipeline's expected input columns.
# Some one-hot categories may be absent in the test split, so they need to be
# added back as zero-filled columns before scoring.
expected_cols = getattr(pipeline, "feature_names_in_", None)
if expected_cols is None and "preprocessor" in getattr(pipeline, "named_steps", {}):
    expected_cols = getattr(pipeline.named_steps["preprocessor"], "feature_names_in_", None)

if expected_cols is not None:
    expected_cols = list(expected_cols)
    missing_cols = [col for col in expected_cols if col not in X_test.columns]
    if missing_cols:
        print(f"Adding {len(missing_cols)} missing feature columns with zeros: {missing_cols}")
    X_test = X_test.reindex(columns=expected_cols, fill_value=0)

print(f"\nTest set     : {X_test.shape[0]:,} variants, {X_test.shape[1]} features")
print(f"Class distribution:\n{y_test.value_counts().rename({0: 'non-pathogenic', 1: 'pathogenic'}).to_string()}")

## 4 — ROC Curve

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(
    y_test, y_proba,
    name=f"Classifier (AUC = {auc:.3f})",
    ax=ax,
    color="steelblue",
)
# Random-classifier baseline
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.500)")
ax.set_title("ROC Curve — Hold-Out Test Set")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "roc_curve.png", dpi=150)
plt.show()
print(f"ROC-AUC = {auc:.4f}  |  saved → reports/roc_curve.png")

## 5 — Precision-Recall Curve

In [ ]:
ap = average_precision_score(y_test, y_proba)
prevalence = y_test.mean()

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba,
    name=f"Classifier (AP = {ap:.3f})",
    ax=ax,
    color="darkorange",
)
# No-skill baseline (prevalence line)
ax.axhline(prevalence, color="k", linestyle="--", lw=1,
           label=f"No-skill baseline (P = {prevalence:.3f})")
ax.set_title("Precision-Recall Curve — Hold-Out Test Set")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "pr_curve.png", dpi=150)
plt.show()
print(f"Average Precision = {ap:.4f}  |  saved → reports/pr_curve.png")

## 6 — Confusion Matrix

In [ ]:
y_pred = pipeline.predict(X_test)
labels = ["Non-pathogenic (0)", "Pathogenic (1)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw counts
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=labels,
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Counts")

# Row-normalised (recall per class)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=labels,
    normalize="true",
    cmap="Blues",
    colorbar=False,
    ax=axes[1],
)
axes[1].set_title("Row-normalised (recall)")

fig.suptitle("Confusion Matrix — Hold-Out Test Set", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(REPORTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/confusion_matrix.png")

## 7 — SHAP Feature Importance

Uses `shap >= 0.45`. A `TreeExplainer` is used for tree-based classifiers (Random Forest, XGBoost, LightGBM); a generic `Explainer` (KernelSHAP) is used as fallback.  
Features are named by extracting them from the `ColumnTransformer` step of the pipeline.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# ── Extract the raw (possibly calibrated) estimator ──────────────────────────
preprocessor = pipeline.named_steps["preprocessor"]
clf_step     = pipeline.named_steps["classifier"]

# Unwrap CalibratedClassifierCV → base estimator
if isinstance(clf_step, CalibratedClassifierCV):
    # CalibratedClassifierCV has .estimator (the base clf) or .calibrated_classifiers_
    base_clf = clf_step.estimator if hasattr(clf_step, "estimator") else clf_step
else:
    base_clf = clf_step

# ── Transform X_test through all pipeline steps except the classifier ─────────
X_test_transformed = preprocessor.transform(X_test)

# ── Recover feature names from ColumnTransformer ─────────────────────────────
try:
    feature_names = preprocessor.get_feature_names_out().tolist()
    # Strip step prefixes added by ColumnTransformer ("num__cadd_score" → "cadd_score")
    feature_names = [n.split("__", 1)[-1] for n in feature_names]
except Exception:
    feature_names = [f"feature_{i}" for i in range(X_test_transformed.shape[1])]

# Convert to DataFrame for nicer SHAP plots
X_test_tr_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# ── Build SHAP explainer ──────────────────────────────────────────────────────
TREE_TYPES = ("RandomForestClassifier", "XGBClassifier", "LGBMClassifier",
              "GradientBoostingClassifier", "ExtraTreesClassifier")

clf_class = type(base_clf).__name__
if clf_class in TREE_TYPES:
    explainer = shap.TreeExplainer(base_clf)
    shap_values = explainer.shap_values(X_test_tr_df)
    # For multi-output tree explainer (RF) shap_values is a list [class0, class1]
    if isinstance(shap_values, list):
        sv = shap_values[1]   # positive class
    else:
        sv = shap_values
else:
    # KernelExplainer fallback — use a background sample for speed
    background = shap.sample(X_test_tr_df, min(100, len(X_test_tr_df)))
    explainer  = shap.KernelExplainer(base_clf.predict_proba, background)
    shap_values = explainer.shap_values(X_test_tr_df, nsamples=100)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f"Explainer type  : {type(explainer).__name__}")
print(f"SHAP matrix     : {sv.shape}")

In [ ]:
# ── Bar plot: mean |SHAP| per feature ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
shap.summary_plot(sv, X_test_tr_df, plot_type="bar", show=False, max_display=20)
plt.title("SHAP — Mean |SHAP value| (top 20 features)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/shap_bar.png")

In [ ]:
# ── Beeswarm plot: value × impact ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(sv, X_test_tr_df, show=False, max_display=20)
plt.title("SHAP Beeswarm — Feature Value vs Impact (top 20)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/shap_beeswarm.png")

## 8 — Per-Gene Pathogenicity Heatmap

Aggregate the predicted pathogenic probability by `Gene × trio_id` and render  
as a clustered heatmap. Genes with consistently high probabilities across trios  
are highlighted in the top rows.

In [ ]:
# Attach predicted probabilities back to the test-set variant rows
df_heat = df_test.copy().reset_index(drop=True)
df_heat["prob_pathogenic"] = y_proba

# Mean probability per (Gene, trio_id)
pivot_df = (
    df_heat.groupby(["Gene", "trio_id"])["prob_pathogenic"]
    .mean()
    .unstack("trio_id")
    .fillna(0)
)

# Keep the top-N most variable genes (cleaner heatmap)
TOP_N = min(40, len(pivot_df))
top_genes = pivot_df.mean(axis=1).nlargest(TOP_N).index
pivot_df = pivot_df.loc[top_genes]

# Cluster rows by mean probability
row_order = pivot_df.mean(axis=1).sort_values(ascending=False).index
pivot_df = pivot_df.loc[row_order]

fig_height = max(6, len(pivot_df) * 0.3)
fig, ax = plt.subplots(figsize=(max(8, len(pivot_df.columns) * 0.8), fig_height))
sns.heatmap(
    pivot_df,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    vmin=0, vmax=1,
    ax=ax,
    cbar_kws={"label": "Mean P(pathogenic)"},
)
ax.set_title(f"Per-Gene Pathogenicity Probability\n(top {TOP_N} genes by mean score)")
ax.set_xlabel("Trio ID")
ax.set_ylabel("Gene")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "gene_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → reports/gene_heatmap.png  ({TOP_N} genes × {pivot_df.shape[1]} trios)")

## 9 — Performance Breakdown by Inheritance Mode

Compute per-group metrics to identify which inheritance patterns the model  
handles well vs. where it struggles (e.g. rare `autosomal_recessive` vs.  
common `de_novo`).

In [ ]:
df_preds = df_test.copy().reset_index(drop=True)
df_preds["y_true"]  = y_test.values
df_preds["y_pred"]  = y_pred
df_preds["y_proba"] = y_proba

rows = []
for mode, grp in df_preds.groupby("inheritance_mode"):
    yt = grp["y_true"].values
    yp = grp["y_pred"].values
    yb = grp["y_proba"].values
    n  = len(yt)
    n_pos = int(yt.sum())

    # Skip groups with only one class for AUC/AP (would raise an error)
    has_both_classes = len(np.unique(yt)) == 2

    row = {
        "inheritance_mode": mode,
        "n_variants":       n,
        "n_pathogenic":     n_pos,
        "prevalence":       round(n_pos / n, 3) if n else float("nan"),
        "f1_macro":         round(f1_score(yt, yp, average="macro",    zero_division=0), 3),
        "f1_weighted":      round(f1_score(yt, yp, average="weighted", zero_division=0), 3),
        "roc_auc":          round(roc_auc_score(yt, yb), 3)          if has_both_classes else float("nan"),
        "avg_precision":    round(average_precision_score(yt, yb), 3) if has_both_classes else float("nan"),
    }
    rows.append(row)

perf_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)

# Export
perf_df.to_csv(REPORTS_DIR / "performance_by_inheritance.csv", index=False)
print("Saved → reports/performance_by_inheritance.csv\n")

# Display styled
perf_df.style \
    .background_gradient(subset=["roc_auc", "avg_precision", "f1_macro"], cmap="RdYlGn", vmin=0, vmax=1) \
    .format({"prevalence": "{:.1%}", "roc_auc": "{:.3f}", "avg_precision": "{:.3f}",
             "f1_macro": "{:.3f}", "f1_weighted": "{:.3f}"}, na_rep="—") \
    .set_caption("Model performance breakdown by inheritance mode")

In [ ]:
# Bar chart of F1-macro per inheritance mode
fig, ax = plt.subplots(figsize=(9, 4))
colors = sns.color_palette("muted", len(perf_df))
bars = ax.bar(perf_df["inheritance_mode"], perf_df["f1_macro"], color=colors)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_xlabel("Inheritance Mode")
ax.set_ylabel("F1 Macro")
ax.set_title("F1 Macro by Inheritance Mode — Hold-Out Test Set")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(REPORTS_DIR / "f1_by_inheritance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/f1_by_inheritance.png")

## 10 — Summary Metrics

Quick summary of all scalar metrics on the hold-out set.

In [ ]:
from src.model import evaluate

metrics = evaluate(pipeline, X_test, y_test, log_to_mlflow=False)

summary = pd.Series({
    "n_test_variants":   metrics["n_test_samples"],
    "accuracy":          round(metrics["accuracy"],          4),
    "f1_macro":          round(metrics["f1_macro"],          4),
    "f1_weighted":       round(metrics["f1_weighted"],       4),
    "roc_auc":           round(metrics["roc_auc"],           4),
    "average_precision": round(metrics["average_precision"], 4),
}, name="hold-out test set")

print(summary.to_string())
print("\nClassification report:\n")
print(metrics["classification_report"])